# Upload Models to HuggingFace Hub

Standalone notebook for uploading trained brain segmentation models to HuggingFace.
Use this when a model was trained and saved to DBFS but never uploaded to HuggingFace.

**Prerequisites:**
- `HUGGING_FACE_TOKEN` env var with write access
- Model files available locally (fetch from DBFS first if needed)

**Models:**

| Model | Local Path | HuggingFace Repo | Status |
|-------|-----------|-----------------|--------|
| Mouse (1,328 classes) | `models/dinov2-upernet-final` | `Noel-Niko/...-mouse` | Already uploaded |
| Human Allen (44 classes) | `models/human-depth3` | `Noel-Niko/...-human` | Already uploaded |
| **Human BigBrain (10 types)** | `models/human-bigbrain` | `Noel-Niko/...-human-bigbrain` | **MISSING** |

**Usage:**
1. Run Cell 0 to check which models are available locally
2. If BigBrain model is missing, run `make fetch-human-bigbrain-from-dbfs` first
3. Run Cell 1 to set your HuggingFace token
4. Run Cell 2 to upload the missing BigBrain model (or all models)

In [ ]:
# Cell 0 — Check which models are available locally
#
# Path resolution order:
#   1. Databricks workspace: /Workspace/.../histology/models/
#   2. Databricks DBFS:      /dbfs/FileStore/allen_brain_data/models/
#   3. Local (repo root):    ./models/
#   4. Local (from notebooks/): ../models/

from pathlib import Path

# --- Auto-detect models directory ---
_model_candidates = [
    Path("/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology/models"),
    Path("/dbfs/FileStore/allen_brain_data/models"),
    Path("models"),
    Path("../models"),
]
MODELS_DIR = next((p for p in _model_candidates if p.exists()), Path("models"))

# --- Auto-detect docs directory ---
_doc_candidates = [
    Path("/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology/docs"),
    Path("docs"),
    Path("../docs"),
]
DOCS_DIR = next((p for p in _doc_candidates if p.exists()), Path("docs"))

print(f"Resolved models dir: {MODELS_DIR.resolve()}")
print(f"Resolved docs dir:   {DOCS_DIR.resolve()}")

# --- DBFS model directory names differ from local names ---
# DBFS uses: final-200ep, human-allen-depth3, human-bigbrain
# Local uses: dinov2-upernet-final, human-depth3, human-bigbrain
_is_dbfs = "dbfs" in str(MODELS_DIR).lower()

MODEL_CONFIGS = {
    "mouse": {
        "local_dir": MODELS_DIR / ("final-200ep" if _is_dbfs else "dinov2-upernet-final"),
        "repo_suffix": "mouse",
        "species_title": "Mouse Brain",
        "num_classes": "1,328",
        "data_source": "Allen Brain Institute CCFv3 10um Nissl staining",
        "metrics": "mIoU: 74.8% (CC), 79.1% (SW)",
        "make_cmd": "make annotate-mouse",
        "download_cmd": "make download-models-mouse",
        "tags": ["brain-segmentation", "histology", "dinov2", "upernet", "neuroscience", "allen-brain-institute"],
        "paper_path": DOCS_DIR / "mouse_paper_draft.md",
    },
    "human": {
        "local_dir": MODELS_DIR / ("human-allen-depth3" if _is_dbfs else "human-depth3"),
        "repo_suffix": "human",
        "species_title": "Human Brain (Allen Depth-3)",
        "num_classes": "44 (depth-3 brain regions)",
        "data_source": "Allen Human Brain Atlas (6 donors, Nissl staining)",
        "metrics": "mIoU: 65.5% (val CC), 65.0% (test SW)",
        "make_cmd": "make annotate-human-allen",
        "download_cmd": "make download-models-human-allen",
        "tags": ["brain-segmentation", "histology", "dinov2", "upernet", "neuroscience", "allen-brain-institute"],
        "paper_path": DOCS_DIR / "human_paper_draft.md",
    },
    "human-bigbrain": {
        "local_dir": MODELS_DIR / "human-bigbrain",
        "repo_suffix": "human-bigbrain",
        "species_title": "Human Brain (BigBrain Tissue Classification)",
        "num_classes": "10 (tissue types)",
        "data_source": "BigBrain 3D histological volume (200um, 9-class tissue classification)",
        "metrics": "mIoU: 60.6% (CC), 61.2% (SW)",
        "make_cmd": "make annotate-human-bigbrain",
        "download_cmd": "make download-models-human-bigbrain",
        "tags": ["brain-segmentation", "histology", "dinov2", "upernet", "neuroscience", "bigbrain"],
        "paper_path": DOCS_DIR / "human_paper_draft.md",
    },
}

# Paper summary snippets embedded in each model card README
PAPER_SUMMARIES = {
    "mouse": """
## Paper

**Transfer Learning for Ultra-Fine-Grained Brain Region Segmentation: An Ablation Study with DINOv2 + UperNet on 1,328 Allen Mouse Brain Atlas Structures**

We conduct a systematic ablation study across 9 training runs on pixel-level segmentation of 1,328 brain structures from the Allen Mouse Brain Atlas CCFv3. The two most impactful interventions are backbone partial fine-tuning (+8.5% mIoU) and extended training from 100 to 200 epochs (+6.0% mIoU). Three attempted improvements — weighted Dice+CE loss, aggressive augmentation, and test-time augmentation — all degrade performance, providing informative negative results.

The proven recipe: (1) DINOv2-Large backbone with last 4/24 blocks unfrozen, (2) differential learning rate (1e-5 backbone, 1e-4 head), (3) plain cross-entropy loss, (4) minimal augmentation, (5) 200 epochs, (6) sliding window evaluation. Per-class segmentation quality is strongly predicted by class pixel count (r=0.794).

See `paper.md` in this repo for the full paper with detailed results, ablation tables, and analysis.
""",
    "human": """
## Paper

**Cross-Species Transfer of Ultra-Fine-Grained Brain Segmentation: From Mouse to Human with DINOv2 + UperNet**

We extend the DINOv2-Large + UperNet approach from mouse (1,328 classes, 79.1% mIoU) to human brain tissue using the Allen Human Brain Atlas (sparse SVG annotations, 6 donors). The depth-3 model (44 brain regions) achieves 65.5% val CC mIoU and 65.0% test SW mIoU with 99.1% pixel accuracy. Major structures (cerebellum, cerebral cortex, thalamus, pons) exceed 99% IoU.

The principal finding is that annotation density, not model capacity, determines the performance ceiling. The training recipe from the mouse ablation study transfers directly to human tissue with no modifications.

See `paper.md` in this repo for the full paper with cross-track comparisons and deployment guidance.
""",
    "human-bigbrain": """
## Paper

**Cross-Species Transfer of Ultra-Fine-Grained Brain Segmentation: From Mouse to Human with DINOv2 + UperNet**

This model is part of a three-track human brain segmentation study. Track B uses the BigBrain 200um classified volume with dense 9-class tissue annotations (Merker stain). The model achieves 60.6% CC mIoU and 61.2% SW mIoU with all 10 classes contributing valid IoU scores.

The BigBrain model serves as a tissue type classifier — distinguishing gray matter, white matter, CSF, meninges, and bone — complementary to the Allen depth-3 model's role as a brain region identifier.

See `paper.md` in this repo for the full paper with cross-track comparisons and deployment guidance.
""",
}

HF_USERNAME = "Noel-Niko"
MODEL_BASE = "dinov2-upernet-20260322-histology-annotation"

REQUIRED_FILES = ["config.json", "preprocessor_config.json"]
WEIGHT_FILES = ["model.safetensors", "pytorch_model.bin"]

print("=" * 65)
print(f"LOCAL MODEL STATUS  (source: {'DBFS' if _is_dbfs else 'local'})")
print("=" * 65)

for name, cfg in MODEL_CONFIGS.items():
    local_dir = cfg["local_dir"]
    repo_id = f"{HF_USERNAME}/{MODEL_BASE}-{cfg['repo_suffix']}"
    exists = local_dir.exists()
    has_configs = exists and all((local_dir / f).exists() for f in REQUIRED_FILES)
    has_weights = exists and any((local_dir / f).exists() for f in WEIGHT_FILES)
    complete = has_configs and has_weights
    paper_exists = cfg["paper_path"].exists()

    status = "READY" if complete else ("INCOMPLETE" if exists else "NOT FOUND")
    icon = "OK" if complete else "!!"  
    print(f"\n  [{icon}] {name}")
    print(f"      Local:  {local_dir}")
    print(f"      Repo:   {repo_id}")
    print(f"      Status: {status}")
    print(f"      Paper:  {'FOUND' if paper_exists else 'NOT FOUND'} ({cfg['paper_path'].name})")
    if exists:
        files = list(local_dir.iterdir())
        total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
        print(f"      Files:  {len(files)} ({total_mb:.0f} MB)")
    elif not _is_dbfs:
        print(f"      Fix:    Run 'make fetch-human-bigbrain-from-dbfs' (or similar)")

print("\n" + "=" * 65)

In [ ]:
# Cell 1 — Configure HuggingFace token
#
# Option A: Set HUGGING_FACE_TOKEN env var before starting Jupyter
# Option B: Paste your token below (delete before committing!)

import os

# Uncomment and paste your token if not set via env var:
# os.environ["HUGGING_FACE_TOKEN"] = "hf_your_token_here"

HF_TOKEN = os.environ.get("HUGGING_FACE_TOKEN")

if HF_TOKEN:
    print(f"HuggingFace token found: {HF_TOKEN[:8]}...")
else:
    print("ERROR: HUGGING_FACE_TOKEN not set.")
    print("Set it with: export HUGGING_FACE_TOKEN=hf_your_token_here")
    print("Or uncomment the line above and paste your token.")

In [ ]:
# Cell 2 — Upload missing model(s) to HuggingFace Hub
#
# Set UPLOAD_SPECIES to control which model(s) to upload:
#   "human-bigbrain"  — Upload only the missing BigBrain model (default)
#   "mouse"           — Upload mouse model
#   "human"           — Upload human Allen model
#   "all"             — Upload all 3 models

from pathlib import Path
from huggingface_hub import HfApi, create_repo

UPLOAD_SPECIES = "human-bigbrain"  # Change to "all" to upload everything

if not HF_TOKEN:
    raise RuntimeError("HUGGING_FACE_TOKEN not set. Run Cell 1 first.")

api = HfApi(token=HF_TOKEN)

if UPLOAD_SPECIES == "all":
    targets = list(MODEL_CONFIGS.keys())
else:
    targets = [UPLOAD_SPECIES]

print(f"Uploading: {targets}")
print("=" * 65)

for species in targets:
    cfg = MODEL_CONFIGS[species]
    local_dir = cfg["local_dir"]
    repo_id = f"{HF_USERNAME}/{MODEL_BASE}-{cfg['repo_suffix']}"

    print(f"\n--- {species} ---")
    print(f"  Local: {local_dir}")
    print(f"  Repo:  {repo_id}")

    if not local_dir.exists():
        print(f"  SKIP: Local directory not found. Fetch from DBFS first.")
        continue

    # Verify model is complete
    has_configs = all((local_dir / f).exists() for f in REQUIRED_FILES)
    has_weights = any((local_dir / f).exists() for f in WEIGHT_FILES)
    if not (has_configs and has_weights):
        print(f"  SKIP: Model incomplete (missing config or weights).")
        continue

    # Create repo
    create_repo(repo_id=repo_id, token=HF_TOKEN, exist_ok=True)
    print(f"  Repo ready: https://huggingface.co/{repo_id}")

    # Upload model files
    files_to_upload = [
        f for f in local_dir.iterdir()
        if f.is_file()
        and f.name not in {"optimizer.pt", "scheduler.pt"}
        and not f.name.startswith("rng_state")
    ]

    print(f"  Uploading {len(files_to_upload)} model files...")
    for filepath in files_to_upload:
        size_mb = filepath.stat().st_size / 1e6
        print(f"    {filepath.name} ({size_mb:.1f} MB)")
        api.upload_file(
            path_or_fileobj=str(filepath),
            path_in_repo=filepath.name,
            repo_id=repo_id,
            token=HF_TOKEN,
        )

    # Upload paper.md (full paper alongside model weights)
    paper_path = cfg["paper_path"]
    if paper_path.exists():
        paper_content = paper_path.read_text(encoding="utf-8")
        api.upload_file(
            path_or_fileobj=paper_content.encode("utf-8"),
            path_in_repo="paper.md",
            repo_id=repo_id,
            token=HF_TOKEN,
        )
        print(f"  Uploaded paper.md ({paper_path.name}, {len(paper_content)} chars)")
    else:
        print(f"  WARNING: Paper not found at {paper_path} — skipping paper.md upload")

    # Build and upload model card README with paper summary
    tags_yaml = "\n".join(f"  - {t}" for t in cfg["tags"])
    paper_summary = PAPER_SUMMARIES.get(species, "")
    model_card = f"""---
license: apache-2.0
tags:
{tags_yaml}
---

# Brain Region Segmentation \u2014 {cfg['species_title']}

DINOv2-Large + UperNet model fine-tuned for semantic segmentation of
brain structures in histological sections.

## Model Details

| Attribute | Value |
|-----------|-------|
| Architecture | DINOv2-Large (304M) + UperNet (38M) |
| Classes | {cfg['num_classes']} |
| Input Size | 518x518 |
| Training Data | {cfg['data_source']} |
| Performance | {cfg['metrics']} |

## Usage

```bash
git clone https://github.com/Noel-Niko/histological-image-analysis
cd histological-image-analysis
make install
{cfg['download_cmd']}
{cfg['make_cmd']} IMAGES=/path/to/your/slides/
```
{paper_summary}
## Citation

If you use this model, please cite the training data sources and the paper included in this repository.

## Repository

Full source code, training notebooks, and all models: https://github.com/Noel-Niko/histological-image-analysis
"""
    api.upload_file(
        path_or_fileobj=model_card.encode("utf-8"),
        path_in_repo="README.md",
        repo_id=repo_id,
        token=HF_TOKEN,
    )
    print(f"  Upload complete: https://huggingface.co/{repo_id}")

print("\n" + "=" * 65)
print("Done!")

In [ ]:
# Cell 3 — Verify upload by checking HuggingFace repos

from huggingface_hub import list_repo_files

print("=" * 65)
print("HUGGINGFACE REPO VERIFICATION")
print("=" * 65)

for species in targets:
    cfg = MODEL_CONFIGS[species]
    repo_id = f"{HF_USERNAME}/{MODEL_BASE}-{cfg['repo_suffix']}"
    print(f"\n  {species}: {repo_id}")
    try:
        files = list_repo_files(repo_id)
        for f in sorted(files):
            print(f"    {f}")
        has_weights = any(f in files for f in WEIGHT_FILES)
        has_config = "config.json" in files
        print(f"    -> {'OK' if has_weights and has_config else 'INCOMPLETE'}")
    except Exception as e:
        print(f"    -> ERROR: {e}")

print("\n" + "=" * 65)